# 20.4 Data & model versioning; model registry

A registry turns model artifacts into governed releases: every candidate has lineage, compatibility, stage, and rollback semantics.

Experiment tracking supplies immutable runs; feature stores supply the input schema that the artifact expects.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(204)


def summarize_registry(workload):
    promoted = workload["candidate_metric"] - workload["champion_metric"] >= workload["tau"]
    truly_bad = workload["true_metric"] < workload["champion_metric"]
    bad_promotion_rate = float(np.mean(promoted & truly_bad))
    release_count = int(workload["candidate_metric"].size)
    seconds = max(float(workload["duration_seconds"]), 1.0)
    return {
        "name": workload["name"],
        "requests": release_count,
        "p50_ms": float(np.percentile(workload["registry_latency_ms"], 50)),
        "p95_ms": float(np.percentile(workload["registry_latency_ms"], 95)),
        "throughput_qps": release_count / seconds,
        "cost_dollars": float(np.sum(workload["cost_usd"])),
        "drift_stat": float(np.mean(workload["schema_mismatch"])),
        "bad_promotion_rate": bad_promotion_rate,
    }


def simulate_registry(name, release_count, noise=0.001, schema_rate=0.0, rollback_rate=0.0):
    champion_metric = RNG.normal(0.804, 0.006, release_count)
    true_lift = RNG.normal(0.004, 0.006, release_count)
    true_metric = champion_metric + true_lift
    candidate_metric = true_metric + RNG.normal(0.0, noise, release_count)
    schema_mismatch = RNG.random(release_count) < schema_rate
    candidate_metric[schema_mismatch] += RNG.uniform(0.002, 0.010, int(np.sum(schema_mismatch)))
    rollback = RNG.random(release_count) < rollback_rate
    registry_latency_ms = RNG.lognormal(mean=np.log(55.0), sigma=0.42, size=release_count)
    cost_usd = 0.0015 + registry_latency_ms * 0.000001
    return {
        "name": name,
        "champion_metric": champion_metric,
        "candidate_metric": candidate_metric,
        "true_metric": true_metric,
        "schema_mismatch": schema_mismatch,
        "rollback": rollback,
        "registry_latency_ms": registry_latency_ms,
        "cost_usd": cost_usd,
        "duration_seconds": max(release_count / 3.0, 1.0),
        "tau": 0.005,
    }


def make_workload_ladder():
    d1 = {
        "name": "D1 tiny champion/candidate registry",
        "champion_metric": np.array([0.804]),
        "candidate_metric": np.array([0.812]),
        "true_metric": np.array([0.812]),
        "schema_mismatch": np.array([False]),
        "rollback": np.array([False]),
        "registry_latency_ms": np.array([44.0]),
        "cost_usd": np.array([0.0015]),
        "duration_seconds": 1.0,
        "tau": 0.005,
    }
    d2 = simulate_registry("D2 1k clean artifacts", 1_000, noise=0.001)
    d3 = simulate_registry("D3 schema changes and metric noise", 10_000, noise=0.006, schema_rate=0.07)
    d4 = simulate_registry("D4 real-ish staged release history", 80_000, noise=0.003, schema_rate=0.03, rollback_rate=0.02)
    d5 = simulate_registry("D5 production-scale registry with rollbacks", 250_000, noise=0.005, schema_rate=0.05, rollback_rate=0.04)
    return [d1, d2, d3, d4, d5]


## The concept, built once: registry promotion gates

The lesson gate is $\Delta=m_{candidate}-m_{champion}$ and promotion requires $\Delta\ge\tau$. We implement the reusable gate and assert $0.812-0.804=0.008\ge0.005$.

In [ ]:

def registry_gate(candidate, champion, tau):
    delta = float(candidate - champion)
    promote = delta >= tau
    return delta, promote

lesson_delta, lesson_promote = registry_gate(0.812, 0.804, 0.005)
assert np.isclose(lesson_delta, 0.008)
assert lesson_promote is True
print("delta", lesson_delta)
print("promote", lesson_promote)


Registry state includes content scope, compatibility, retention, and rollback pointers. The lesson names $1000$ rows plus $12$ schema fields and $5\times180=900$ MB retention.

In [ ]:

rows = 1000
schema_fields = 12
missing_inputs = 24 - 24
retention_mb = 5 * 180
rollback_pointer_change = 7 - 6
assert rows == 1000
assert schema_fields == 12
assert missing_inputs == 0
assert retention_mb == 900
assert rollback_pointer_change == 1
print("hash scope", rows, "rows and", schema_fields, "fields")
print("retention MB", retention_mb)


## The dataset ladder

Family F17 has no shared ML-training ladder here. We build the operational workload ladder inline: D1 tiny trace, D2 larger volume, D3 spikes or drift, D4 real-ish synthetic trace, and D5 production-scale simulation.

In [ ]:

workloads = make_workload_ladder()
for workload in workloads:
    print(workload["name"])
    print("shape", workload["candidate_metric"].shape)
    print("sample deltas", np.round((workload["candidate_metric"] - workload["champion_metric"])[:5], 4))


## Run the same method across D1-D5

The metric is bad-promotion rate: promoted candidates whose true metric is below the champion. The table also keeps registry latency, release throughput, storage-like cost, and schema drift visible.

In [ ]:

workloads = make_workload_ladder()
summaries = [summarize_registry(workload) for workload in workloads]

header = f"{'rung':<42} {'load':>10} {'p50':>9} {'p95':>9} {'cost':>10} {'qps':>10} {'bad_promotion_rate':>12}"
print(header)
for row in summaries:
    line = f"{row['name']:<42} {row['requests']:>10} {row['p50_ms']:>9.2f} {row['p95_ms']:>9.2f} {row['cost_dollars']:>10.3f} {row['throughput_qps']:>10.2f} {row['bad_promotion_rate']:>12.5f}"
    print(line)


The first row is the operational artifact: p95 latency, total cost, and throughput by rung. The second plot is the lesson metric versus load.

In [ ]:

names = [row["name"].split()[0] for row in summaries]
loads = np.array([row["requests"] for row in summaries], dtype=float)
p95_values = np.array([row["p95_ms"] for row in summaries], dtype=float)
cost_values = np.array([row["cost_dollars"] for row in summaries], dtype=float)
throughput_values = np.array([row["throughput_qps"] for row in summaries], dtype=float)
metric_values = np.array([row["bad_promotion_rate"] for row in summaries], dtype=float)
cost_per_request = cost_values / loads
normalized_p95 = p95_values / max(float(np.max(p95_values)), 1.0)
normalized_cost = cost_per_request / max(float(np.max(cost_per_request)), 1.0)
normalized_throughput = throughput_values / max(float(np.max(throughput_values)), 1.0)

fig = plt.figure(figsize=(15, 7))
grid = fig.add_gridspec(2, 5, height_ratios=[1.0, 1.15])
for index, name in enumerate(names):
    ax = fig.add_subplot(grid[0, index])
    values = [normalized_p95[index], normalized_cost[index], normalized_throughput[index]]
    ax.bar(["p95", "cost", "qps"], values)
    ax.set_ylim(0.0, 1.05)
    ax.set_title(name)
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylabel("normalized")
summary_ax = fig.add_subplot(grid[1, :])
summary_ax.plot(loads, metric_values, marker="o")
summary_ax.set_xscale("log")
summary_ax.set_title("bad-promotion rate vs load")
summary_ax.set_xlabel("records or requests")
summary_ax.set_ylabel("bad-promotion rate")
fig.suptitle("Per-workload latency, cost, throughput, and metric-vs-load")
fig.tight_layout()


## Pitfall on D5: versioning paths instead of content

A file path can stay fixed while rows or schema change. The fix is a content signature that includes both row content and schema contract before promotion.

In [ ]:

d5 = workloads[-1]
path_only_detects = np.zeros_like(d5["schema_mismatch"], dtype=bool)
content_hash_detects = d5["schema_mismatch"]
undetected_path_only = int(np.sum(d5["schema_mismatch"] & (~path_only_detects)))
undetected_content_hash = int(np.sum(d5["schema_mismatch"] & (~content_hash_detects)))
print("undetected with path-only versioning", undetected_path_only)
print("undetected with content hash", undetected_content_hash)


## Evaluate it + Practice

- Metric: bad-promotion rate; no-skill baseline promotes every candidate above an absolute metric without champion comparison.
- Sanity check: D1 delta clears the threshold.
- Ablation: remove schema compatibility and observe bad promotions rise under schema mismatch.
- Failure signals: rollbacks cluster after promotion, schema drift rises, or content hashes change under the same path.

Practice: raise $\tau$ from $0.005$ to $0.010$ and recompute promotions.

Practice: simulate a rollback by moving the served pointer from version 7 to version 6.

Practice: estimate storage cost for keeping 10 versions at 180 MB each.